# Run mirror-leech-telegram-bot in Google Colab

This notebook is intended for testing or short-lived runs. Colab can disconnect or recycle its runtime at any time, and files on `/content` are temporary. Use a persistent host for a continuously available bot. Configure a MongoDB `DATABASE_URL` if you want bot settings to survive a runtime reset. Keep Telegram credentials private; do not share this notebook with credentials in cell output.

In [ ]:
!git clone https://github.com/anasty17/mirror-leech-telegram-bot.git /content/mirror-leech-telegram-bot
%cd /content/mirror-leech-telegram-bot

In [ ]:
!apt-get update -qq
!apt-get install -y -qq aria2 ffmpeg p7zip-full qbittorrent-nox sabnzbdplus rclone
!python -m pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import socket
import subprocess
import time

root = Path.cwd()
downloads = Path("/app/downloads")
downloads.mkdir(parents=True, exist_ok=True)

# Aria2 RPC is bound to loopback; the bot connects to port 6800.
aria_log = open("/content/aria2-colab.log", "w")
aria2 = subprocess.Popen([
    "aria2c", "--enable-rpc=true", "--rpc-listen-port=6800",
    "--rpc-listen-all=false", "--rpc-allow-origin-all=true",
    f"--dir={downloads}", "--file-allocation=none", "--console-log-level=warn",
], stdout=aria_log, stderr=subprocess.STDOUT, start_new_session=True)

# qBittorrent's Web API has no login on loopback only.
profile = Path("/content/qbittorrent-profile")
config_dir = profile / "qBittorrent" / "config"
config_dir.mkdir(parents=True, exist_ok=True)
qbit_config = (root / "qBittorrent" / "config" / "qBittorrent.conf").read_text()
qbit_config = "\n".join(
    "WebUI\\Address=127.0.0.1" if line.startswith("WebUI\\Address=")
    else "WebUI\\LocalHostAuth=false" if line.startswith("WebUI\\LocalHostAuth=")
    else line
    for line in qbit_config.splitlines()
)
(config_dir / "qBittorrent.conf").write_text(qbit_config)
qbit_log = open("/content/qbittorrent-colab.log", "w")
qbittorrent = subprocess.Popen([
    "qbittorrent-nox", f"--profile={profile}", "--webui-port=8090",
    "--confirm-legal-notice",
], stdout=qbit_log, stderr=subprocess.STDOUT, start_new_session=True)

# Use a temporary SABnzbd config with its API restricted to loopback.
sab_config = (root / "sabnzbd" / "SABnzbd.ini").read_text()
sab_config = sab_config.replace("host = ::", "host = 127.0.0.1")
sab_config = sab_config.replace("/usr/src/app/downloads", str(downloads))
sab_config_path = Path("/content/SABnzbd-colab.ini")
sab_config_path.write_text(sab_config)
sab_log = open("/content/sabnzbd-colab.log", "w")
sabnzbd = subprocess.Popen([
    "sabnzbdplus", "-f", str(sab_config_path), "-s", "127.0.0.1:8070",
], stdout=sab_log, stderr=subprocess.STDOUT, start_new_session=True)

def wait_for_service(process, port, name):
    for _ in range(60):
        if process.poll() is not None:
            raise RuntimeError(f"{name} exited during startup; check its /content/*-colab.log file")
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                return
        except OSError:
            time.sleep(1)
    raise TimeoutError(f"{name} did not start listening on port {port}")

wait_for_service(aria2, 6800, "Aria2")
wait_for_service(qbittorrent, 8090, "qBittorrent")
wait_for_service(sabnzbd, 8070, "SABnzbd")
import json
from urllib.request import Request, urlopen

aria_request = Request(
    "http://127.0.0.1:6800/jsonrpc",
    data=json.dumps({"jsonrpc": "2.0", "id": "colab", "method": "aria2.getVersion"}).encode(),
    headers={"Content-Type": "application/json"},
)
with urlopen(aria_request, timeout=5) as response:
    json.load(response)
with urlopen("http://127.0.0.1:8090/api/v2/app/version", timeout=5) as response:
    if response.status != 200:
        raise RuntimeError("qBittorrent Web API did not return HTTP 200")
with urlopen("http://127.0.0.1:8070/sabnzbd/api?mode=version&output=json&apikey=mltb", timeout=5) as response:
    json.load(response)
print("Aria2, qBittorrent, and SABnzbd are running on loopback.")

## Enter Telegram credentials

Create a bot token with [@BotFather](https://t.me/BotFather), and obtain your Telegram API ID and hash from [my.telegram.org](https://my.telegram.org). Inputs below are hidden and are only kept in this runtime's environment. Optionally enter a MongoDB connection string to persist bot settings.

In [ ]:
import os
from getpass import getpass

for key in ("BOT_TOKEN", "OWNER_ID", "TELEGRAM_API", "TELEGRAM_HASH"):
    value = getpass(f"{key}: ").strip()
    if not value:
        raise ValueError(f"{key} is required")
    os.environ[key] = value

database_url = getpass("DATABASE_URL (optional, press Enter to skip): ").strip()
if database_url:
    os.environ["DATABASE_URL"] = database_url

In [ ]:
!python -m bot